In [ ]:
import kagglehub
from seaborn import heatmap

# Download latest version
path = kagglehub.dataset_download("yelp-dataset/yelp-dataset")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd


In [ ]:
import seaborn as sns
import  matplotlib.pyplot as plt


In [ ]:
reviews = pd.concat(
    pd.read_json(f'{path}/yelp_academic_dataset_review.json',
                 lines=True, chunksize=100_000, nrows=500_000)
)

In [ ]:
business = pd.concat(
    pd.read_json(f'{path}/yelp_academic_dataset_business.json',
                 lines=True, chunksize=100_000, nrows=500_000)
)

In [ ]:
user = pd.concat(
    pd.read_json(f'{path}/yelp_academic_dataset_user.json',
                 lines=True, chunksize=100_000, nrows=500_000)
)

In [ ]:
tip = pd.concat(
    pd.read_json(f'{path}/yelp_academic_dataset_tip.json',
                 lines=True, chunksize=100_000, nrows=500_000)
)

In [ ]:
checkin = pd.read_json(f'{path}/yelp_academic_dataset_checkin.json',
                       lines=True, convert_dates=False)

In [ ]:
(reviews == '').sum() + (reviews == 'None').sum()


In [ ]:
(checkin == '').sum() + (checkin == 'None').sum()


In [ ]:

(user == '').sum() + (user == 'None').sum()

In [ ]:

(tip == '').sum() + (tip == 'None').sum()

In [ ]:

(business == '').sum() + (business == 'None').sum()

In [ ]:
import  numpy as np
business["missingAddress"] = business["address"].isin(['', 'None']).astype(int)
business["missingpostal_code"]= business["postal_code"].isin(['', 'None']).astype(int)

In [ ]:
business["categories"].value_counts().reset_index().sort_values("count",ascending=False)

In [ ]:
business

In [ ]:
reviews=reviews.drop(columns=["cool","funny"])

In [ ]:
reviews

In [ ]:
user=user.drop(columns=["funny","cool","elite","friends","fans","compliment_cool","compliment_funny","compliment_plain","compliment_photos","compliment_profile","compliment_cute"])

In [ ]:
user


In [ ]:
business = business.dropna(subset=["categories"]).copy()

In [ ]:
business["cat_list"] = business["categories"].str.split(", ")
exploded = business.explode("cat_list")
exploded["cat_list"] = exploded["cat_list"].str.strip()

In [ ]:
YELP_TO_FSQ = {
    # ── spot types ──
    "Bakeries":                    (13002, "Bakery"),
    "Bars":                        (13003, "Bar"),
    "Breakfast & Brunch":          (13028, "Breakfast Spot"),
    "Cafes":                       (13032, "Café"),
    "Coffee & Tea":                (13034, "Coffee Shop"),
    "Bubble Tea":                  (13033, "Bubble Tea Shop"),
    "Desserts":                    (13040, "Dessert Shop"),
    "Ice Cream & Frozen Yogurt":   (13046, "Ice Cream Parlor"),
    "Donuts":                      (13043, "Donut Shop"),
    "Juice Bars & Smoothies":      (13059, "Juice Bar"),
    "Fast Food":                   (13145, "Fast Food Restaurant"),
    "Food Court":                  (13052, "Food Court"),
    "Food Trucks":                 (13054, "Food Truck"),
    "Pizza":                       (13064, "Pizzeria"),
    "Steakhouses":                 (13383, "Steakhouse"),

    # ── cuisines ──
    "American (New)":              (13068, "American"),
    "American (Traditional)":      (13068, "American"),
    "Asian Fusion":                (13072, "Asian"),
    "Chinese":                     (13099, "Chinese"),
    "Indian":                      (13199, "Indian"),
    "Italian":                     (13236, "Italian"),
    "Japanese":                    (13263, "Japanese"),
    "Korean":                      (13289, "Korean"),
    "Mediterranean":               (13302, "Mediterranean"),
    "Mexican":                     (13303, "Mexican"),
    "Middle Eastern":              (13306, "Middle Eastern"),
    "Thai":                        (13352, "Thai"),
    "Vietnamese":                  (13377, "Vietnamese"),
    "Seafood":                     (13338, "Seafood"),
    "Sushi Bars":                  (13276, "Sushi"),
    "Vegan":                       (13385, "Vegan / Vegetarian"),
    "Vegetarian":                  (13385, "Vegan / Vegetarian"),
}

print(f"{len(YELP_TO_FSQ)} Yelp category names mapped")


In [ ]:
exploded["fsqid"]    = exploded["cat_list"].map(lambda c: YELP_TO_FSQ.get(c, (None, None))[0])
exploded["fsq_name"] = exploded["cat_list"].map(lambda c: YELP_TO_FSQ.get(c, (None, None))[1])

In [ ]:
mapped = exploded.dropna(subset=["fsqid"])

In [ ]:
mapped

In [ ]:
reviews["wordsInDesc"]=reviews["text"].str.split().str.len()

In [ ]:
reviews.loc[reviews["wordsInDesc"].between(1,4),"text"]

In [23]:
from ftlangdetect import detect as ft_detect

def fast_lang(text):
    if not isinstance(text, str) or len(text.strip()) < 20:
        return None
    return ft_detect(text.replace("\n", " "))["lang"]

reviews["lang"] = reviews["text"].map(fast_lang)   # minutes, not hours
reviews = reviews[reviews["lang"] == "en"]

In [25]:
reviews.loc[reviews["wordsInDesc"].between(3,24),"text"]

11        Locals recommended Milktooth, and it's an amaz...
22        I thoroughly enjoyed the show.  Chill way to s...
25        Went for lunch. Beef brisket sandwich was awes...
26        Best thai food in the area.  Everything was au...
27        Service was crappy, and food was mediocre.  I ...
                                ...                        
499956    Awesome mexican food here!  I was sad to hear ...
499966    Fast, Fantastic & had a key blank that 7 other...
499975    omg love the pizza here !! it's like eating ol...
499988    Amazing cheese cake, cookie crumble! Will defi...
499990    The standard wrap sandwich at this store was d...
Name: text, Length: 46821, dtype: str

In [26]:
reviews_3UP = reviews[reviews["wordsInDesc"]>=25]

In [27]:
reviews_3UP

,review_id,user_id,business_id,stars,useful,text,date,wordsInDesc,lang
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11,101,en
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18,151,en
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30,55,en
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03,40,en
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15,94,en
...,...,...,...,...,...,...,...,...,...
499995,q4Z6NCR1IMRdpPorTD89Tg,tCPzvPwsd_DwBx6whsDlLQ,-QwHN9KoluPcA0YFllwFYQ,5,0,We won the Playoff game last night and is sche...,2021-06-30 16:14:03,127,en
499996,KTg0i04SbVWGWHHJmGOI_A,rj0uSTXu1rPVgAfOMHYltQ,t_v2TyjeqaRkrfZKudY9cA,5,0,We have been a resident almost 4 years at Manz...,2021-03-24 20:21:27,64,en
499997,ZuMNAPcArFtaGufe-nwGOA,MVqzYt-E7y1Mdw9F_7nLjw,3XirYkP9PJvVXIEDPNNXLA,4,0,This place is hyped as one of the best places ...,2021-07-03 04:51:07,54,en
499998,Lohri9uZyvoNnH5z_NT6DA,vn777Y2vCynYYUYJjNYdYg,gfPDLZimZu1NtBIDbeXetg,2,1,Saw the reviews so thought I'd try this place....,2019-04-16 23:28:41,145,en


In [29]:
checkin


,business_id,date
0,---kPU91CF4Lq2-WlRu9Lw,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020..."
1,--0iUa4sNDFiZFrAdIWhZQ,"2010-09-13 21:43:09, 2011-05-04 23:08:15, 2011..."
2,--30_8IhuyMHbSOcNWd6DQ,"2013-06-14 23:29:17, 2014-08-13 23:20:22"
3,--7PUidqRWpRSpXebiyxTg,"2011-02-15 17:12:00, 2011-07-28 02:46:10, 2012..."
4,--7jw19RH9JKXgFohspgQw,"2014-04-21 20:42:11, 2014-04-28 21:04:46, 2014..."
...,...,...
131925,zznJox6-nmXlGYNWgTDwQQ,"2013-03-23 16:22:47, 2013-04-07 02:03:12, 2013..."
131926,zznZqH9CiAznbkV6fXyHWA,2021-06-12 01:16:12
131927,zzu6_r3DxBJuXcjnOYVdTw,"2011-05-24 01:35:13, 2012-01-01 23:44:33, 2012..."
131928,zzw66H6hVjXQEt0Js3Mo4A,"2016-12-03 23:33:26, 2018-12-02 19:08:45"


In [33]:
checkin["checkinCount"] = checkin["date"].str.split(",").str.len()
checkin

,business_id,date,checkinCount
0,---kPU91CF4Lq2-WlRu9Lw,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020...",11
1,--0iUa4sNDFiZFrAdIWhZQ,"2010-09-13 21:43:09, 2011-05-04 23:08:15, 2011...",10
2,--30_8IhuyMHbSOcNWd6DQ,"2013-06-14 23:29:17, 2014-08-13 23:20:22",2
3,--7PUidqRWpRSpXebiyxTg,"2011-02-15 17:12:00, 2011-07-28 02:46:10, 2012...",10
4,--7jw19RH9JKXgFohspgQw,"2014-04-21 20:42:11, 2014-04-28 21:04:46, 2014...",26
...,...,...,...
131925,zznJox6-nmXlGYNWgTDwQQ,"2013-03-23 16:22:47, 2013-04-07 02:03:12, 2013...",67
131926,zznZqH9CiAznbkV6fXyHWA,2021-06-12 01:16:12,1
131927,zzu6_r3DxBJuXcjnOYVdTw,"2011-05-24 01:35:13, 2012-01-01 23:44:33, 2012...",23
131928,zzw66H6hVjXQEt0Js3Mo4A,"2016-12-03 23:33:26, 2018-12-02 19:08:45",2


In [34]:
split_dates = checkin["date"].str.split(", ")

checkin["firstCheckin"]  = pd.to_datetime(split_dates.str[0])
checkin["latestCheckin"] = pd.to_datetime(split_dates.str[-1])
checkin

,business_id,date,checkinCount,firstCheckin,latestCheckin
0,---kPU91CF4Lq2-WlRu9Lw,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020...",11,2020-03-13 21:10:56,2021-11-11 16:23:50
1,--0iUa4sNDFiZFrAdIWhZQ,"2010-09-13 21:43:09, 2011-05-04 23:08:15, 2011...",10,2010-09-13 21:43:09,2014-04-12 23:04:47
2,--30_8IhuyMHbSOcNWd6DQ,"2013-06-14 23:29:17, 2014-08-13 23:20:22",2,2013-06-14 23:29:17,2014-08-13 23:20:22
3,--7PUidqRWpRSpXebiyxTg,"2011-02-15 17:12:00, 2011-07-28 02:46:10, 2012...",10,2011-02-15 17:12:00,2015-09-27 13:18:32
4,--7jw19RH9JKXgFohspgQw,"2014-04-21 20:42:11, 2014-04-28 21:04:46, 2014...",26,2014-04-21 20:42:11,2021-06-21 19:59:50
...,...,...,...,...,...
131925,zznJox6-nmXlGYNWgTDwQQ,"2013-03-23 16:22:47, 2013-04-07 02:03:12, 2013...",67,2013-03-23 16:22:47,2021-05-16 21:43:26
131926,zznZqH9CiAznbkV6fXyHWA,2021-06-12 01:16:12,1,2021-06-12 01:16:12,2021-06-12 01:16:12
131927,zzu6_r3DxBJuXcjnOYVdTw,"2011-05-24 01:35:13, 2012-01-01 23:44:33, 2012...",23,2011-05-24 01:35:13,2013-12-13 00:58:14
131928,zzw66H6hVjXQEt0Js3Mo4A,"2016-12-03 23:33:26, 2018-12-02 19:08:45",2,2016-12-03 23:33:26,2018-12-02 19:08:45


In [35]:
checkin=checkin.drop(columns=["date"])

In [36]:
checkin

,business_id,checkinCount,firstCheckin,latestCheckin
0,---kPU91CF4Lq2-WlRu9Lw,11,2020-03-13 21:10:56,2021-11-11 16:23:50
1,--0iUa4sNDFiZFrAdIWhZQ,10,2010-09-13 21:43:09,2014-04-12 23:04:47
2,--30_8IhuyMHbSOcNWd6DQ,2,2013-06-14 23:29:17,2014-08-13 23:20:22
3,--7PUidqRWpRSpXebiyxTg,10,2011-02-15 17:12:00,2015-09-27 13:18:32
4,--7jw19RH9JKXgFohspgQw,26,2014-04-21 20:42:11,2021-06-21 19:59:50
...,...,...,...,...
131925,zznJox6-nmXlGYNWgTDwQQ,67,2013-03-23 16:22:47,2021-05-16 21:43:26
131926,zznZqH9CiAznbkV6fXyHWA,1,2021-06-12 01:16:12,2021-06-12 01:16:12
131927,zzu6_r3DxBJuXcjnOYVdTw,23,2011-05-24 01:35:13,2013-12-13 00:58:14
131928,zzw66H6hVjXQEt0Js3Mo4A,2,2016-12-03 23:33:26,2018-12-02 19:08:45
